In [1]:
# Question 1: Logic Warmup
from logic import Expr
A = Expr('A')
B = Expr('B')
C = Expr('C')
D = Expr('D')

expr1 = C % (B | D)           # C ↔ (B ∨ D)
expr2 = ~A >> (~B & ~D)       # A → (¬B ∧ ¬D)
expr3 = ~(B & ~C) >> A        # ¬(B ∧ ¬C) → A
expr4 = ~D >> C               # ¬D → C

logic_warmup = expr1 & expr2 & expr3 & expr4
print("Question1: Logic Warmup:\n", logic_warmup)

Question1: Logic Warmup:
 ((((C ↔ (B ∨ D)) ∧ (¬A → (¬B ∧ ¬D))) ∧ (¬(B ∧ ¬C) → A)) ∧ (¬D → C))


In [2]:
# Question 2: Logic Workout
from functools import reduce
from itertools import combinations

def at_least_one(vars):
    #Returns an Expr that is true if at least one variable in the list is true.
    return reduce(lambda a, b: a | b, vars)

def at_most_one(vars):
    #Returns an Expr that is true if at most one variable in the list is true.
    # For each pair, add ¬xi ∨ ¬xj
    clauses = [~vi | ~vj for vi, vj in combinations(vars, 2)]
    # Chain binary AND across all pairwise clauses
    return reduce(lambda a, b: a & b, clauses)

def exact_one(vars):
    return at_least_one(vars) & at_most_one(vars)

A = Expr('A')
B = Expr('B')
C = Expr('C')

exact_one_expr = exact_one([A, B, C])
print("Question 2: Logic Workout:\n",exact_one_expr)

Question 2: Logic Workout:
 (((A ∨ B) ∨ C) ∧ (((¬A ∨ ¬B) ∧ (¬A ∨ ¬C)) ∧ (¬B ∨ ¬C)))


In [3]:
# Question 3: PAC Physics and Satisfiability
import pycosat
from logic import Expr, And, Or, Not, to_cnf, ExactOne
# Maze dimensions and wall set
width, height = 3, 3
walls = {(1,1)}     # set of (x,y) coordinates that are walls
max_time = 3        # number of time steps to model

# Helper constructors
def position(x, y, t):
    """Pacman is at (x,y) at time t."""
    return Expr(f"P_{x}_{y}_{t}")

def action(direction, t):
    """Pacman takes 'direction' at time t."""
    return Expr(f"{direction}_{t}")

def is_free(x, y, walls):
    """True if (x,y) is not a wall."""
    return (x, y) not in walls

clauses = []

#Pacman must be at exactly one non‐wall position at each time t
for t in range(max_time + 1):
    pos_atoms = [position(x, y, t)
                 for x in range(width) for y in range(height)
                 if is_free(x, y, walls)]
    clauses.append( ExactOne(pos_atoms) )

#Pacman must take exactly one action at each time step t (for t=0…max_time–1)
directions = ["N", "S", "E", "W"]
for t in range(max_time):
    act_atoms = [action(d, t) for d in directions]
    clauses.append( ExactOne(act_atoms) )

#Movement rules: if Pacman was at (x,y) at t and takes the given action, and the target cell is free, then he must be at the neighbor at t+1.
for t in range(max_time):
    for x in range(width):
        for y in range(height):
            if not is_free(x, y, walls):
                continue
            # North
            nx, ny = x, y+1
            if 0 <= ny < height and is_free(nx, ny, walls):
                clauses.append(
                    Or(Not(position(x, y, t)),
                       Not(action("N", t)),
                       position(nx, ny, t+1))
                )
            # South
            nx, ny = x, y-1
            if 0 <= ny < height and is_free(nx, ny,walls):
                clauses.append(
                    Or(Not(position(x, y, t)),
                       Not(action("S", t)),
                       position(nx, ny, t+1))
                )
            # East
            nx, ny = x+1, y
            if 0 <= nx < width and is_free(nx, ny, walls):
                clauses.append(
                    Or(Not(position(x, y, t)),
                       Not(action("E", t)),
                       position(nx, ny, t+1))
                )
            # West
            nx, ny = x-1, y
            if 0 <= nx < width and is_free(nx, ny, walls):
                clauses.append(
                    Or(Not(position(x, y, t)),
                       Not(action("W", t)),
                       position(nx, ny, t+1))
                )
# Convert all clauses to CNF
cnf_clauses = [to_cnf(c) for c in clauses]

#first few CNF clauses
for c in cnf_clauses[:5]:
    print(f"{c}")

# example parameters
x, y, t = 1, 1, 2
# example position and action
pre  = Expr(f"P_{x}_{y}_{t-1}")      # P_1_1_1
act  = Expr(f"W_{t-1}")              # W_1
post = Expr(f"P_{x-1}_{y}_{t}")      # P_0_1_2
rule = And(pre, act) >> post
# convert to CNF
cnf_rule = to_cnf(rule)
print("Example movement rule:\no If PAC is at (x, y) at time t-1, takes action WEST, and (x-1, y) is not a wall, then PAC is at (x-1, y) at time t.\n", cnf_rule)



((((((((P_0_0_0 ∨ P_0_1_0) ∨ P_0_2_0) ∨ P_1_0_0) ∨ P_1_2_0) ∨ P_2_0_0) ∨ P_2_1_0) ∨ P_2_2_0) ∧ ((((((((((((((((((((((((((((¬P_0_0_0 ∨ ¬P_0_1_0) ∧ (¬P_0_0_0 ∨ ¬P_0_2_0)) ∧ (¬P_0_0_0 ∨ ¬P_1_0_0)) ∧ (¬P_0_0_0 ∨ ¬P_1_2_0)) ∧ (¬P_0_0_0 ∨ ¬P_2_0_0)) ∧ (¬P_0_0_0 ∨ ¬P_2_1_0)) ∧ (¬P_0_0_0 ∨ ¬P_2_2_0)) ∧ (¬P_0_1_0 ∨ ¬P_0_2_0)) ∧ (¬P_0_1_0 ∨ ¬P_1_0_0)) ∧ (¬P_0_1_0 ∨ ¬P_1_2_0)) ∧ (¬P_0_1_0 ∨ ¬P_2_0_0)) ∧ (¬P_0_1_0 ∨ ¬P_2_1_0)) ∧ (¬P_0_1_0 ∨ ¬P_2_2_0)) ∧ (¬P_0_2_0 ∨ ¬P_1_0_0)) ∧ (¬P_0_2_0 ∨ ¬P_1_2_0)) ∧ (¬P_0_2_0 ∨ ¬P_2_0_0)) ∧ (¬P_0_2_0 ∨ ¬P_2_1_0)) ∧ (¬P_0_2_0 ∨ ¬P_2_2_0)) ∧ (¬P_1_0_0 ∨ ¬P_1_2_0)) ∧ (¬P_1_0_0 ∨ ¬P_2_0_0)) ∧ (¬P_1_0_0 ∨ ¬P_2_1_0)) ∧ (¬P_1_0_0 ∨ ¬P_2_2_0)) ∧ (¬P_1_2_0 ∨ ¬P_2_0_0)) ∧ (¬P_1_2_0 ∨ ¬P_2_1_0)) ∧ (¬P_1_2_0 ∨ ¬P_2_2_0)) ∧ (¬P_2_0_0 ∨ ¬P_2_1_0)) ∧ (¬P_2_0_0 ∨ ¬P_2_2_0)) ∧ (¬P_2_1_0 ∨ ¬P_2_2_0)))
((((((((P_0_0_1 ∨ P_0_1_1) ∨ P_0_2_1) ∨ P_1_0_1) ∨ P_1_2_1) ∨ P_2_0_1) ∨ P_2_1_1) ∨ P_2_2_1) ∧ ((((((((((((((((((((((((((((¬P_0_0_1 ∨ ¬P_0_1_1) ∧ (¬P_0_0_1 ∨ ¬P_0_2_1)) ∧ (¬P_0_0_1

In [4]:
# Question 4: Path Planning with Logic
def cnf_to_dimacs(expr, var_map):
    def flatten_and(e):
        if e.op == 'AND':
            out = []
            for arg in e.args:
                out.extend(flatten_and(arg))
            return out
        return [e]

    def flatten_or(e):
        if e.op == 'OR':
            out = []
            for arg in e.args:
                out.extend(flatten_or(arg))
            return out
        return [e]

    dimacs = []
    for clause in flatten_and(expr):
        lits = []
        for lit in flatten_or(clause):
            if lit.op == 'NOT':
                v, sign = lit.args[0], -1
            else:
                v, sign = lit, 1
            vid = var_map.setdefault(v, len(var_map) + 1)
            lits.append(sign * vid)
        dimacs.append(lits)
    return dimacs

# Plan Path, given a grid of width x height, walls, start and goal positions
def plan_path(width, height, walls, start, goal, max_depth=10):
    sx, sy = start
    gx, gy = goal
    dx = abs(gx - sx)
    dy = abs(gy - sy)

    for depth in range(1, max_depth + 1):
        clauses = []

        # start at (sx,sy,0)
        clauses.append(to_cnf(position(sx, sy, 0)))

        #exactly‑one‑position for each t
        for t in range(depth + 1):
            pos_atoms = [
                position(x, y, t)
                for x in range(width)
                for y in range(height)
                if is_free(x, y, walls)
            ]
            clauses.append(to_cnf(ExactOne(pos_atoms)))

        #for each t=0…T, add actions + movement rules
        for t in range(depth):
            # 3a) exactly‑one‑action
            acts = [action(d, t) for d in ("N","S","E","W")]
            clauses.append(to_cnf(ExactOne(acts)))

            # movement rules + invalid‑move bans
            for x in range(width):
                for y in range(height):
                    if not is_free(x, y, walls):
                        continue
                    pre = position(x, y, t)
                    for d, (dx_, dy_) in [
                        ("N", (0, 1)),
                        ("S", (0,-1)),
                        ("E", (1, 0)),
                        ("W", (-1,0))
                    ]:
                        act = action(d, t)
                        nx, ny = x + dx_, y + dy_

                        if 0 <= nx < width and 0 <= ny < height and is_free(nx, ny, walls):
                            post = position(nx, ny, t+1)
                            # valid move:
                            clauses.append(to_cnf( Or(Not(pre), Not(act), post) ))
                        else:
                            # invalid move:
                            clauses.append(to_cnf( Or(Not(pre), Not(act)) ))

            if t < dy:
                #ban EAST
                clauses.append(to_cnf( Not(action("E", t)) ))
            elif t < dy + dx:
                #ban NORTH
                clauses.append(to_cnf( Not(action("N", t)) ))

        #force goal at t=depth
        clauses.append(to_cnf(position(gx, gy, depth)))

        # build DIMACS + solve
        var_map = {}
        dimacs  = []
        for c in clauses:
            dimacs.extend(cnf_to_dimacs(c, var_map))

        sol = pycosat.solve(dimacs)
        if sol != "UNSAT":
            # extract plan & actions
            plan    = []
            actions = []
            for t in range(depth+1):
                for x in range(width):
                    for y in range(height):
                        if not is_free(x, y, walls):
                            continue
                        v = position(x, y, t)
                        if var_map[v] in sol:
                            plan.append((x, y))
                            break
            for t in range(depth):
                for d in ("N","S","E","W"):
                    v = action(d, t)
                    if var_map[v] in sol:
                        actions.append(d)
                        break
            return plan, actions

    return None

# Example

walls       = {(1,1)}
width, height = 3, 3
start, goal = (0,0), (2,2)
path, acts  = plan_path(width, height, walls, start, goal, max_depth=10)
print("Task: For t=1, 2, …:\n• Check for a feasible assignment such that (xt, yt) is the goal.\n• If found, return the path.\n• Add constraints:\no at ∈ {N, S, E, W}\no If at= N and (xt, yt+1) is not a wall, then (xt+1, yt+1) = (xt, yt+1)\n")
print("Pacman Start and Goal", start, goal)
print("Positions:", path)
print("Actions:  ", acts)

Task: For t=1, 2, …:
• Check for a feasible assignment such that (xt, yt) is the goal.
• If found, return the path.
• Add constraints:
o at ∈ {N, S, E, W}
o If at= N and (xt, yt+1) is not a wall, then (xt+1, yt+1) = (xt, yt+1)

Pacman Start and Goal (0, 0) (2, 2)
Positions: [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2)]
Actions:   ['N', 'N', 'E', 'E']


In [10]:
# Question 5: Eating All Food
# Helpers function for current positions action and food
def food_sym(x, y, t):
    return Expr(f"F_{x}_{y}_{t}")

def plan_with_food(width, height, walls, food, start, goal, max_depth=10):
    sx, sy = start
    gx, gy = goal

    for depth in range(1, max_depth + 1):
        clauses = []

        # Start at (sx,sy,0)
        clauses.append(to_cnf(position(sx, sy, 0)))

        # Exactly‑one‑position at each t
        for t in range(depth + 1):
            pos_atoms = [
                position(x, y, t)
                for x in range(width)
                for y in range(height)
                if is_free(x, y, walls)
            ]
            clauses.append(to_cnf(ExactOne(pos_atoms)))

        # Exactly‑one‑action + movement rules
        for t in range(depth):
            #one action
            acts = [action(d, t) for d in ("N","S","E","W")]
            clauses.append(to_cnf(ExactOne(acts)))

            for x in range(width):
                for y in range(height):
                    if not is_free(x, y, walls):
                        continue
                    pre = position(x, y, t)
                    for d, (dx, dy) in [
                        ("N",(0, 1)),("S",(0,-1)),
                        ("E",(1, 0)),("W",(-1,0))
                    ]:
                        act = action(d, t)
                        nx, ny = x + dx, y + dy
                        if 0 <= nx < width and 0 <= ny < height and is_free(nx, ny, walls):
                            clauses.append(to_cnf(
                                Or(Not(pre), Not(act), position(nx, ny, t+1))
                            ))
                        else:
                            clauses.append(to_cnf(
                                Or(Not(pre), Not(act))
                            ))

        #Initial food at t=0
        for (fx, fy) in food:
            clauses.append(to_cnf(food_sym(fx, fy, 0)))

        # Food dynamics for t=0…depth-1
        for t in range(depth):
            for (fx, fy) in food:
                F_t   = food_sym(fx, fy, t)
                F_t1  = food_sym(fx, fy, t+1)
                P_ft  = position(fx, fy, t)
                clauses.append(to_cnf(
                    Or(Not(F_t), P_ft, F_t1)
                ))
                clauses.append(to_cnf(
                    Or(Not(F_t), Not(P_ft), Not(F_t1))
                ))

        #All food eaten by t=depth
        for (fx, fy) in food:
            clauses.append(to_cnf(
                Not(food_sym(fx, fy, depth))
            ))

        # Force goal at (gx,gy,depth)
        clauses.append(to_cnf(position(gx, gy, depth)))

        # Build DIMACS + solve
        var_map = {}
        dimacs  = []
        for c in clauses:
            dimacs.extend(cnf_to_dimacs(c, var_map))

        sol = pycosat.solve(dimacs)
        if sol != "UNSAT":
            # Extract plan & actions
            plan    = []
            actions = []
            for t in range(depth+1):
                for x in range(width):
                    for y in range(height):
                        if not is_free(x, y, walls):
                            continue
                        v = position(x, y, t)
                        if var_map[v] in sol:
                            plan.append((x, y))
                            break
            for t in range(depth):
                for d in ("N","S","E","W"):
                    v = action(d, t)
                    if var_map[v] in sol:
                        actions.append(d)
                        break
            return plan, actions

    return None

# Example

walls       = {(1,1)}
width, height = 3, 3
start, goal = (0,0), (2,2)
# Suppose there is food initially at (0,2), (2,0), and (2,2)
food = {(0,2), (2,0), (2,2)}

path, acts = plan_with_food(
    width, height, walls,
    food, start, goal,
    max_depth=20
)
print("Objective: Use logical representation to capture food dynamics.")
print("Pacman Start and Goal", start, goal)
print("Foods are at ", food)
print("Positions:", path)
print("Actions:  ", acts)

Objective: Use logical representation to capture food dynamics.
Pacman Start and Goal (0, 0) (2, 2)
Foods are at  {(0, 2), (2, 0), (2, 2)}
Positions: [(0, 0), (1, 0), (2, 0), (2, 1), (2, 2), (1, 2), (0, 2), (1, 2), (2, 2)]
Actions:   ['E', 'E', 'N', 'N', 'W', 'W', 'E', 'E']


In [6]:
# Question 6: Localization

def localize(width, height, walls, actions, sensor_signals):
    T = len(actions)
    clauses = []

    # Exactly‐one‐position at every t=0…T
    for t in range(T+1):
        pos_atoms = [
            position(x, y, t)
            for x in range(width)
            for y in range(height)
            if is_free(x, y, walls)
        ]
        clauses.append(to_cnf(ExactOne(pos_atoms)))

    #action and movement encoding for t=0…T−1
    for t in range(T):
        # force the observed action, forbid the others
        for d in ("N","S","E","W"):
            lit = action(d, t)
            clauses.append(to_cnf(lit) if d == actions[t] else to_cnf(Not(lit)))

        # movement rules + invalid‐move bans
        for x in range(width):
            for y in range(height):
                if (x,y) in walls: 
                    continue
                pre = position(x, y, t)
                for d,(dx,dy) in [("N",(0,1)),("S",(0,-1)),
                                  ("E",(1,0)),("W",(-1,0))]:
                    act = action(d, t)
                    nx, ny = x+dx, y+dy
                    if 0 <= nx < width and 0 <= ny < height and (nx,ny) not in walls:
                        # legal move
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act), position(nx, ny, t+1))
                        ))
                    else:
                        # ban illegal move
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act))
                        ))

    # Sensor constraints for t=0…T
    for t in range(T+1):
        sig = sensor_signals[t]
        for x in range(width):
            for y in range(height):
                if (x,y) in walls: 
                    continue
                for d,(dx,dy) in [("N",(0,1)),("S",(0,-1)),("E",(1,0)),("W",(-1,0))]:
                    nx, ny = x+dx, y+dy
                    wall_exists = not (0<=nx<width and 0<=ny<height) or ((nx,ny) in walls)
                    if sig[d] != wall_exists:
                        clauses.append(to_cnf(Not(position(x, y, t))))

    # build the single KB in DIMACS form
    var_map = {}
    dimacs  = []
    for c in clauses:
        dimacs.extend(cnf_to_dimacs(c, var_map))
    possible = []
    for t in range(T+1):
        S_t = set()
        for x in range(width):
            for y in range(height):
                if (x,y) in walls:
                    continue
                vid = var_map[position(x, y, t)]
                if pycosat.solve(dimacs + [[vid]]) != "UNSAT":
                    S_t.add((x, y))
        possible.append(S_t)

    return possible

# Example

actions = ['N','N','E','E']

sensor_signals = [
  # t=0 @ (0,0)
  {'N': False, 'S': True,  'E': False, 'W': True},
  # t=1 @ (0,1)
  {'N': False, 'S': False, 'E': True,  'W': True},
  # t=2 @ (0,2)
  {'N': True,  'S': False, 'E': False, 'W': True},
  # t=3 @ (1,2)
  {'N': True,  'S': True,  'E': False, 'W': False},
  # t=4 @ (2,2)
  {'N': True,  'S': False, 'E': True,  'W': False},
]

print("Objective: Localize PACMAN based on sensor signals.\nTask:\n• PACMAN does not know its own position.\n• The sensor tells whether there is a wall in the NSEW directions.\n• Given a sequence of actions and sensor signals, infer PACMAN’s position.")
beliefs = localize(3, 3, {(1,1)}, actions, sensor_signals)
for t, S in enumerate(beliefs):
    print(f"t={t}: {sorted(S)}")

Objective: Localize PACMAN based on sensor signals.
Task:
• PACMAN does not know its own position.
• The sensor tells whether there is a wall in the NSEW directions.
• Given a sequence of actions and sensor signals, infer PACMAN’s position.
t=0: [(0, 0)]
t=1: [(0, 1)]
t=2: [(0, 2)]
t=3: [(1, 2)]
t=4: [(2, 2)]


In [7]:
#Question 7: Mapping
def wall(x, y):
    return Expr(f"W_{x}_{y}")

def is_in_grid(x, y, width, height):
    return 0 <= x < width and 0 <= y < height

def map_walls(width, height, start, actions, sensor_signals):
    sx, sy = start
    T = len(actions)

    clauses = []

    #Pac‑Man starts at (sx,sy,0)
    clauses.append(to_cnf(position(sx, sy, 0)))

    # Exactly‑one‑position at each t=0…T
    for t in range(T+1):
        pos_atoms = [
            position(x, y, t)
            for x in range(width)
            for y in range(height)
        ]
        clauses.append(to_cnf(ExactOne(pos_atoms)))

    #Action assertions + movement & wall consistency
    for t in range(T):
        for d in ("N","S","E","W"):
            lit = action(d, t)
            clauses.append(to_cnf(lit) if d == actions[t] else to_cnf(Not(lit)))

        # movement implies no wall at target + update position
        for x in range(width):
            for y in range(height):
                pre = position(x, y, t)
                for d, (dx, dy) in [("N",(0,1)),("S",(0,-1)),("E",(1,0)),("W",(-1,0))]:
                    act = action(d, t)
                    nx, ny = x + dx, y + dy

                    # If the move stays in‑grid:
                    if is_in_grid(nx, ny, width, height):
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act), Not(wall(nx, ny)))
                        ))
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act), position(nx, ny, t+1))
                        ))
                    else:
                        # invalid move
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act))
                        ))

    #Sensor → wall constraints 
    for t in range(T+1):
        sig = sensor_signals[t]
        for x in range(width):
            for y in range(height):
                pre = position(x, y, t)
                for d, (dx, dy) in [("N",(0,1)),("S",(0,-1)),
                                    ("E",(1,0)),("W",(-1,0))]:
                    nx, ny = x + dx, y + dy
                    # Only constrain if neighbor is in‑grid; off-grid is implicitly wall.
                    if is_in_grid(nx, ny, width, height):
                        if sig[d]:
                            # saw a wall → W(nx,ny) must be true
                            clauses.append(to_cnf(
                                Or(Not(pre), wall(nx, ny))
                            ))
                        else:
                            # saw no wall → W(nx,ny) must be false
                            clauses.append(to_cnf(
                                Or(Not(pre), Not(wall(nx, ny)))
                            ))

    #Build DIMACS
    var_map = {}
    dimacs  = []
    for c in clauses:
        dimacs.extend(cnf_to_dimacs(c, var_map))

    #Solve once for the combined KB
    model = pycosat.solve(dimacs)
    if model == "UNSAT":
        raise ValueError("No consistent map and trajectory found!")

    # Extract wall assignments
    inv_map = {v:k for k,v in var_map.items()}
    walls = set()
    for lit in model:
        if lit > 0:
            e = inv_map[lit]
            if isinstance(e, Expr) and e.op.startswith("W_"):
                # positive W_x_y means there is a wall
                x,y = map(int, e.op.split("_")[1:])
                walls.add((x,y))

    return walls


width, height = 3,3
start = (0,0)
actions = ['N','N','E','E']
sensor_signals = [
  {'N':False,'S':True, 'E':False,'W':True},   # t=0
  {'N':False,'S':False,'E':True, 'W':True},   # t=1
  {'N':True, 'S':False,'E':False,'W':True},   # t=2
  {'N':True, 'S':True, 'E':False,'W':False},  # t=3
  {'N':True, 'S':False,'E':True, 'W':False},  # t=4
]
print("Objective: Map the walls of the maze.\nTask:\n• PACMAN knows its starting position.\n• It has a sensor that reports whether there is a wall on NSEW.\n• Use this to reconstruct the map layout over time based on PACMAN’s movements\nand sensor readings")

print("Pacman Start", start)
print("sensor signals:  ", sensor_signals)
inferred_walls = map_walls(width, height, start, actions, sensor_signals)
print("Inferred wall cells:", inferred_walls)

Objective: Map the walls of the maze.
Task:
• PACMAN knows its starting position.
• It has a sensor that reports whether there is a wall on NSEW.
• Use this to reconstruct the map layout over time based on PACMAN’s movements
and sensor readings
Pacman Start (0, 0)
sensor signals:   [{'N': False, 'S': True, 'E': False, 'W': True}, {'N': False, 'S': False, 'E': True, 'W': True}, {'N': True, 'S': False, 'E': False, 'W': True}, {'N': True, 'S': True, 'E': False, 'W': False}, {'N': True, 'S': False, 'E': True, 'W': False}]
Inferred wall cells: {(1, 1)}


In [8]:
#Question 8: Simultaneous Localization and Mapping (SLAM)
from itertools import combinations

def ExactK(vars, k):
    n = len(vars)
    # impossible
    if k < 0 or k > n:
        return Expr('FALSE')
    # at most k: forbid any subset of size k+1 all true
    if k == n:
        at_most = Expr('TRUE')
    else:
        at_most = And(*[
            Or(*[Not(v) for v in comb])
            for comb in combinations(vars, k+1)
        ])
    # at least k: require every subset of size (n-k+1) has at least one true
    if k == 0:
        at_least = Expr('TRUE')
    else:
        at_least = And(*[
            Or(*comb)
            for comb in combinations(vars, n - k + 1)
        ])
    return And(at_least, at_most)

def slam(width, height, start, actions, wall_counts):
    sx, sy = start
    T = len(actions)
    clauses = []
    clauses.append(to_cnf(position(sx, sy, 0)))

    # Exactly‐one‐position per t
    for t in range(T+1):
        pos_atoms = [position(x,y,t) for x in range(width) for y in range(height)]
        clauses.append(to_cnf(ExactOne(pos_atoms)))

    # Actions + movement + no‐wall at target
    for t in range(T):
        for d in ("N","S","E","W"):
            lit = action(d,t)
            clauses.append(to_cnf(lit if d==actions[t] else Not(lit)))

        for x in range(width):
            for y in range(height):
                pre = position(x,y,t)
                for d,(dx,dy) in [("N",(0,1)),("S",(0,-1)),
                                  ("E",(1,0)),("W",(-1,0))]:
                    act = action(d,t)
                    nx, ny = x+dx, y+dy
                    if is_in_grid(nx, ny, width, height):
                        # movement implies next position & no wall there
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act), position(nx,ny,t+1))
                        ))
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act), Not(wall(nx,ny)))
                        ))
                    else:
                        # invalid move
                        clauses.append(to_cnf(
                            Or(Not(pre), Not(act))
                        ))

    # Sensor‐count: account for off‐grid walls
    for t in range(T+1):
        k = wall_counts[t]
        for x in range(width):
            for y in range(height):
                pre = position(x,y,t)
                nbr_vars = []
                offgrid = 0
                for dx,dy in [(0,1),(0,-1),(1,0),(-1,0)]:
                    nx, ny = x+dx, y+dy
                    if is_in_grid(nx, ny, width, height):
                        nbr_vars.append(wall(nx, ny))
                    else:
                        offgrid += 1
                k_eff = k - offgrid
                if not (0 <= k_eff <= len(nbr_vars)):
                    raise ValueError(f"Impossible wall_count {k} at t={t}, cell=({x},{y})")
                ek = ExactK(nbr_vars, k_eff)
                clauses.append(to_cnf(Or(Not(pre), ek)))

    # Build DIMACS and solve
    var_map = {}
    dimacs = []
    for c in clauses:
        dimacs.extend(cnf_to_dimacs(c, var_map))

    sol = pycosat.solve(dimacs)
    if sol == "UNSAT":
        raise ValueError("No solution")

    # 6) Extract trajectory and walls
    inv = {v:k for k,v in var_map.items()}
    traj = {}
    walls_set = set()
    for lit in sol:
        if lit > 0:
            e = inv[lit]
            if e.op.startswith("P_"):
                _,x,y,t = e.op.split("_")
                traj[int(t)] = (int(x), int(y))
            if e.op.startswith("W_"):
                _,x,y = e.op.split("_")
                walls_set.add((int(x), int(y)))

    return traj, walls_set

# Example

width, height = 3,3
start = (0,0)
actions = ['N','N','E','E']
wall_counts = [2,2,2,2,2]

traj, walls = slam(width, height, start, actions, wall_counts)
print("Trajectory:", traj)
print("Inferred walls:", walls)

Trajectory: {0: (0, 0), 1: (0, 1), 2: (0, 2), 3: (1, 2), 4: (2, 2)}
Inferred walls: {(1, 1)}
